In [11]:
from dotenv import load_dotenv
import os

load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
APPLICATIONINSIGHTS_CONNECTION_STRING = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")

In [12]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import (
    CoherenceEvaluator,
)

try:
    credential = DefaultAzureCredential()
    token = credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    print(ex)
    

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
)

In [13]:
def test_coherence_evaluator(query, response):
    coherence_evaluator = CoherenceEvaluator(model_config=model_config)
    score = coherence_evaluator(
        query=query, 
        response=response
    )
    return score


In [33]:
import os
import logging

from opentelemetry._logs import set_logger_provider
from opentelemetry.sdk._logs import (
    LoggerProvider,
    LoggingHandler,
)
from opentelemetry.sdk._logs.export import BatchLogRecordProcessor
from azure.monitor.opentelemetry.exporter import AzureMonitorLogExporter


def set_up_logging():
    logger_provider = LoggerProvider()
    set_logger_provider(logger_provider)

    exporter = AzureMonitorLogExporter(connection_string=os.environ["APPLICATIONINSIGHTS_CONNECTION_STRING"])
    logger_provider.add_log_record_processor(BatchLogRecordProcessor(exporter))

    # Attach LoggingHandler to namespaced logger
    handler = LoggingHandler()
    logger = logging.getLogger(__name__)
    logger.addHandler(handler)
    logger.setLevel(logging.NOTSET)
    return logger, logger_provider

In [34]:
# This must be done before any other telemetry calls
logger, logger_provider = set_up_logging()

In [35]:
# query="What is the capital of France?"
# response="Paris is the capital of France."
# score = test_coherence_evaluator(query, response)
# #add custom properties to the logs 
# llm_properties = {
#     'custom_dimensions': {
#         'llm_version': AZURE_OPENAI_API_VERSION,
#         'llm_model': AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
#         'temperature': 0,
#         'test_id': 'test_coherence_evaluator',
#         'query': query,
#         'response': response,
#         'score': score
        
#     }
# }
# logger.warning("Test", extra=llm_properties)

logger.debug("DEBUG: Debug with properties", extra={"debug": "true"})
logger_provider.force_flush()

True